# Run 000 — Exploratory Data Analysis**Amazon ML Challenge 2024 — entity value extraction from product images.**Predict `"<number> <unit>"` for a given (image, `entity_name`) pair.Scored by **F1 with exact string match**, so `"2 gram"` is correct while`"2.0 gram"`, `"2 gms"` and `"2 g"` are all wrong.This notebook runs **before any training**. It is CPU-only and takes a couple ofminutes. Its job is to answer the questions that decide what we build:1. Distribution over `entity_name` — the label distribution2. Unit mix within each entity3. Empty / missing `entity_value` rate4. Numeric value distribution per entity (log scale, outliers flagged)5. `group_id` cardinality and long-tail shape6. **Value string format audit** — the one that predicts post-processing work7. Duplicate `image_link` count — how many images we actually need to downloadAll logic lives in `src/amlc24/data/eda.py`; this notebook only calls it anddisplays the result.

## 1. Setup

In [ ]:
# Locate the repo and put its `src/` on sys.path.## Three layouts are supported, in priority order:#   1. the repo uploaded as a Kaggle Dataset  -> /kaggle/input/<slug>/.../src#   2. the repo git-cloned into the session   -> /kaggle/working/Amazon-ML-24/src#   3. running locally from the repo itself## Nothing below hardcodes a dataset slug: paths.py does the environment# detection, and this cell only has to find the package.import sys, glob, osfrom pathlib import Pathdef find_src():    candidates = []    for root in sorted(glob.glob("/kaggle/input/*/")):        candidates += [Path(root) / "src", *[Path(p) for p in glob.glob(root + "*/src")]]    candidates += [Path(p) for p in sorted(glob.glob("/kaggle/working/*/src"))]    candidates += [Path.cwd() / "src", Path.cwd().parent / "src"]    for c in candidates:        if (c / "amlc24" / "__init__.py").exists():            return c.resolve()    return NoneSRC = find_src()if SRC is None:    # Not mounted anywhere: clone it (Kaggle notebooks have internet enabled).    !git clone --depth 1 https://github.com/MurtuzaShaikh26/Amazon-ML-24.git /kaggle/working/Amazon-ML-24    SRC = Path("/kaggle/working/Amazon-ML-24/src")sys.path.insert(0, str(SRC))print("src:", SRC)import amlc24from amlc24.logging_utils import setup_loggingfrom amlc24.paths import describesetup_logging()print("amlc24", amlc24.__version__)for k, v in describe().items():    print(f"  {k:20s} {v}")

## 2. Run the profile`run_eda()` loads `train.csv`, computes every table, saves them to`results/eda/`, and renders the charts.

In [ ]:
from amlc24.pipeline.run_eda import run_edaresult = run_eda(plots=True)profile = result["profile"]print(f"\nProfiled {result['n_rows']:,} rows")print("Tables:", ", ".join(profile))print("Saved to:", result["out_dir"])

## 3. Label distribution over `entity_name`The primary label distribution: how many rows of each entity type, and how manyunits each one permits.

In [ ]:
import pandas as pdpd.set_option("display.width", 200)pd.set_option("display.max_rows", 200)display(profile["entity_distribution"])

In [ ]:
from amlc24.data.eda import plot_entity_distributionfrom amlc24.data.load import load_traindf = load_train()plot_entity_distribution(df);

## 4. Unit distribution within each entityWhich units actually appear for each entity, and whether they are in the allowedlist from `constants.py`. A unit outside that list is an **invalid prediction**,so any that show up here in the *labels* are important to know about.

In [ ]:
display(profile["unit_distribution"])

### Cross-tab (entity × unit)

In [ ]:
display(profile["unit_crosstab"])

In [ ]:
from amlc24.data.eda import plot_unit_distributionplot_unit_distribution(df);

### Allowed units per entity, from the dataset's `constants.py`

In [ ]:
display(profile["allowed_units"])

## 5. Empty / missing `entity_value`Empty labels are **not** noise — the metric scores a correct empty prediction asa true negative. The empty rate is therefore the share of the eval set we canscore on by correctly abstaining.

In [ ]:
display(profile["empty_rate"])

In [ ]:
from amlc24.data.eda import plot_empty_rateplot_empty_rate(df);

## 6. Numeric value distribution per entityMin / median / max plus robust (MAD-based) outlier counts. These distributionsare heavy-tailed, so a standard-deviation outlier rule would be useless here.

In [ ]:
display(profile["value_stats"])

In [ ]:
from amlc24.data.eda import plot_value_histogramsplot_value_histograms(df);

## 7. `group_id` distributionHow many distinct product categories, how big the largest is, and how long thetail runs.

In [ ]:
display(profile["group_distribution"])display(profile["group_top"])

In [ ]:
from amlc24.data.eda import plot_group_tailplot_group_tail(df);

## 8. Value string format audit — read this one carefullyBecause the metric is **exact string match**, formatting is worth as much as thenumber. This table is the specification for`postprocess/normalize.py::format_number`:* if `trailing_dot_zero` is ~0%, predictions must never emit `.0`* if `thousands_separator` is ~0%, never emit commas* `range_value` tells us how often the `range_rule` will fire* `unit_outside_allowed_list` and `invalid_unit_but_canonicalisable` size the  alias table's job

In [ ]:
display(profile["format_audit"])

### Concrete examples of each awkward format

In [ ]:
display(profile["format_examples"])

## 9. Duplicate imagesOne product photo often carries several entities, so the number of images todownload is below the row count. `download_saving_pct` is how much thededuplicated download saves.

In [ ]:
display(profile["image_duplication"])display(profile["entities_per_image"].head(15))

## 10. Observations<!-- Fill this in after reading the tables above. -->